In [ ]:
import umap
import pickle
import torch
import numpy as np
import networkx as nx
from networkx.algorithms import community as cd
import community.community_louvain as community_louvain

nft2emb_unaware = pickle.load(
        open(f"../embeddings/nft2emb_dummyViT-train.pkl", 'rb'))

# "entropia" dei copiati

In [ ]:
import pandas as pd
nft2category = pickle.load(open(f"../input/nft2category.pkl", "rb"))
edges=pd.read_csv("../graphs_(15-11)/Monthly/COLL_Level/directed_smart_pool_mean(max)/unaware/Top k 100 Min_sim 0.0/(2021, 3)/edge_list.csv")
edges=edges[edges.weight>.7]
csv=pd.read_csv("/home/user/Scrivania/Untitled.csv")
csv.index=csv.Id
csv_dict=csv.to_dict()

G=nx.from_pandas_edgelist(edges,edge_attr=True)
# G=nx.from_pandas_edgelist(edges,edge_attr=True,create_using=nx.DiGraph)
nft2comm=community_louvain.best_partition(G)

# edges["community_id"]=edges["target"].apply(lambda nft: csv_dict["modularity_class"][nft])
edges["community_id"]=edges["target"].apply(lambda nft: nft2comm[nft])


In [ ]:
len(set(nft2comm.values())),len(set(edges["community_id"].values))

In [ ]:
def outDegree(df, source):
    return len(df[df["source"]==source])
def inDegree(df, target):
    return len(df[df["target"]==target])
def outDegreeW(df, source):
    return df[df["source"]==source].weight.sum()
def inDegreeW(df, target):
    return df[df["target"]==target].weight.sum()
def coll(nft):
    return eval(nft)[0]
def coll(nft):
    return nft
def stratifyGraph(community):
    


    STRAT={}
    N=0
    while len(community)>0:
        
        nodes=list(set(community.source).union(community.target))
        outDegs={}
        for node in nodes:
            outDegs[node]=outDegree(community,node)
        stratification={}
        for diffdeg in list(outDegs.values()):
            stratification[diffdeg]=[node for node in nodes if outDegs[node]==diffdeg]

        STRAT[N]=stratification[0]
       
        # display(community[community.target.isin(stratification[0])])
        community=community.drop(community[community.target.isin(stratification[0])].index)
        N+=1
        # break
    STRAT= {k:v for k,v in sorted(STRAT.items(),key=lambda x : x[0])}
    return STRAT


In [ ]:
# community2info={}


# edges_dict=edges.to_dict()
# comm2top_nfts={}
# for c,community in edges.groupby("community_id"):
#     nft2in_out={}
#     for nft,_ in community.groupby("source"):
#         nft2in_out[nft]={"in":inDegree(community,nft),"out":outDegree(community,nft)}
    
#     nft2in_out={k:v for k,v in sorted(nft2in_out.items(),key=lambda x : x[1]["in"],reverse=True)}

#     nodes=community.target.values
#     colls=[eval(nft)[0] for nft in nodes]
#     colls,freqs=np.unique(colls,return_counts=True)
#     freqs=freqs/freqs.sum()
#     coll2ptc={colls[i]:freqs[i] for i in range(len(colls))}
#     coll2ptc={k:v for k,v in sorted(coll2ptc.items(),key=lambda x : x[1],reverse=True)}
#     community["coll"]=community.target.apply(lambda x: eval(x)[0])
#     coll2nfts={coll:community_coll.target.values for coll, community_coll in community.groupby("coll")}

#     comm2top_nfts[0]=[]
#     community2info[c]={
#         "coll2ptc":coll2ptc,
#         "coll2nfts":coll2nfts,
#         "nft2in_out":nft2in_out
#     }


# top_copiati={community:list(community_info["coll2ptc"].items())[0] for community,community_info in community2info.items()}
# top_copiati={community:(collection,weight) for community,(collection,weight) in top_copiati.items()}

# ispiratori={}
# weighted=True
# for k,v in top_copiati.values():
#     if not weighted:
#         v=1
#     if k in ispiratori:
#         ispiratori[k]=ispiratori[k]+v
#     else:
#         ispiratori[k]=v

# #prendo la prima top collection
# nft_top_copiati={community:community_info["coll2nfts"][list(community_info["coll2ptc"].keys())[0]] for community,community_info in community2info.items()}

# # unique,counts=np.unique([k for k,v in top_copiati],return_counts=True)

# # {k:v for k,v in sorted(dict(zip(unique,counts)).items(),key=lambda x : x[1],reverse=True)}
# {k:v for k,v in sorted(ispiratori.items(),key=lambda x : x[1],reverse=True)}
# #concateno tutti gli embedding degli nft copiati per ogni community id
# embs_top_copiati={id:torch.cat([nft2emb_unaware[nft].unsqueeze(0) for nft in nfts],dim=0) for id,nfts in nft_top_copiati.items()}

# #calcolo la matrice delle distanze del coseno fra i top copiati all'interno di ogni community mantennendo solo la matrice triangolare inferiore 
# entropy_top_copiati={id:torch.tril(torch.cat([dist(emb,embs).unsqueeze(0) for emb in embs]),diagonal=-1) for id,embs in embs_top_copiati.items()}

# import matplotlib.pyplot as plt
# plt.imshow(entropy_top_copiati[23])
# #faccio la media dei soli elementi di interesse
# entropy_top_copiati={id:embs[embs!=0].mean()  for id,embs in entropy_top_copiati.items()}
# # se solo un nft era fonte di ispirazione la distanza è nan in quanto la media delle distanze opera su una matrice delle distanze vuota
# entropy_top_copiati={id:entropy.item() if not torch.isnan(entropy)  else 1.0  for id,entropy in entropy_top_copiati.items()}


In [ ]:
COMM2STRAT={}
for c,community in edges.groupby("community_id"):
    stratification=stratifyGraph(community)
    COMM2STRAT[c]=stratification
    
    for strato,nfts in stratification.items():
        print(len(set([coll(nft) for nft in nfts])))
    print()
    # print(stratification[3])
    # print(outDegree(community,stratification[1][0]))
    # display(community)
    # break

In [ ]:
community2info={}
PERCENTILE=75
N_STD=.5
dist=torch.nn.CosineSimilarity(dim=1)
def getMeanPDist(x1,x2):
    mean_sim=torch.tril(torch.cat([dist(emb,x1).unsqueeze(0) for emb in x2]),diagonal=-1)
    mean_sim=mean_sim[mean_sim!=0].mean()
    mean_sim=mean_sim.item() if not torch.isnan(mean_sim)  else 1.0
    return mean_sim
    
edges_dict=edges.to_dict()
comm2top_nfts={}
for c,community in edges.groupby("community_id"):
    nft2in_out={}
    relevant={}
    non_relevant={}
    nodes=list(set(community.source.values).union(set(community.target.values)))
    for nft in nodes:
        nft2in_out[nft]={"in":inDegree(community,nft),"out":outDegree(community,nft),"inW":inDegreeW(community,nft),"outW":outDegreeW(community,nft)}

    for nft in nft2in_out:
        nft2in_out[nft]["relevance"]=nft2in_out[nft]["inW"] if nft2in_out[nft]["outW"]==0 else nft2in_out[nft]["inW"]/nft2in_out[nft]["outW"]

    relevances=[nft["relevance"] for nft in nft2in_out.values()]


    relevance_value=max(np.mean(relevances),np.percentile(relevances, PERCENTILE))
    # relevance_value=np.mean(relevances)+N_STD*np.std(relevances)

    for nft in nft2in_out:
        if nft2in_out[nft]["relevance"]>=relevance_value:

            relevant[nft]=nft2in_out[nft]["relevance"]
        else:
            non_relevant[nft]=nft2in_out[nft]["relevance"]
    
    nft2in_out={k:v for k,v in sorted(nft2in_out.items(),key=lambda x : x[1]["in"],reverse=True)}


    embs_relevant=torch.cat([nft2emb_unaware[nft].unsqueeze(0) for nft in relevant],dim=0)
    mean_sim_relevant=getMeanPDist(embs_relevant,embs_relevant)

    try:
        embs_non_relevant=torch.cat([nft2emb_unaware[nft].unsqueeze(0) for nft in non_relevant],dim=0)
        mean_sim_non_relevant=getMeanPDist(embs_non_relevant,embs_non_relevant)
        mean_sim_rel_non_rel=getMeanPDist(embs_relevant,embs_non_relevant)
    except:
        mean_sim_non_relevant=0.0
        mean_sim_rel_non_rel=0.0


    g=nx.from_pandas_edgelist(community,edge_attr=True,create_using=nx.DiGraph)
    # communities=community_louvain.best_partition(g)

    community2info[c]={
        "nodes":nodes,
        "community":community,
        "nx_comm":g,
        "nft2in_out":nft2in_out,
        "relevant":relevant,
        "isTree":nx.is_tree(g),
        "collections_in_relevant":list(set([eval(nft)[0] for nft in relevant])),
        "category_in_relevant":list(set([nft2category[nft] for nft in relevant])),
        "non_relevant":non_relevant,
        "collections_in_non_relevant":list(set([eval(nft)[0] for nft in non_relevant])),
        "category_in_non_relevant":list(set([nft2category[nft] for nft in non_relevant])),
        "relevance_value":relevance_value,
        "mean_sim_relevant":mean_sim_relevant,
        "mean_sim_non_relevant":mean_sim_non_relevant,
        "mean_sim_rel_non_rel":mean_sim_rel_non_rel
    }



In [ ]:
community2info[21]["relevance_value"],community2info[21]["nft2in_out"]

In [ ]:
nx.is_forest(community2info[21]["nx_comm"])

# Commenti sui valori 
* community_id id della comunità considerata
* relevance_value valore di relevance oltre la quale si considera un nft come RILEVANTE. La relevance è definita come $rel(nft)=\frac{inDegW(nft)}{outDegW(nft)}$ se $OutDegW(nft) \neq 0 altrimenti $rel(nft)=1$
* |Relevant| numero di nft tali per cui $rel(nft)>$ relevance_value
* |Non Relevant| numero di nft tali per cui $rel(nft)<$ relevance_value
* |Colls Relevant| numero di collections presenti in Relevant
* |Colls Non Relevant| numero di collections presenti in Non Relevant
* |Cat Relevant| numero di categorie presenti in Relevant
* |Cat Non Relevant| numero di categorie presenti in Non Relevant
* Mean Sim between Relevant $mean(cos\_dist(x1,x2)) \forall$ x1,x2 $\in$ Relevant 
* Mean Sim between Relevant $mean(cos\_dist(x1,x2)) \forall$ x1,x2 $\in$ Non Relevant 
* Mean Sim between Relevant -Non Relevant  $mean(cos\_dist(x1,x2)) \forall$ x1 $\in$ Relevant,x2 $\in$ Non Relevant 

In [ ]:
records=[]

for community_id, community_info in community2info.items():
    record={
        "community_id":community_id,
        "relevance_value":community_info["relevance_value"],
        "|Relevant|":len(community_info["relevant"]),
        "|Non Relevant|":len(community_info["non_relevant"]),
        "|Colls(Relevant)|":len(community_info["collections_in_relevant"]),
        "|Colls(Non Relevant)|":len(community_info["collections_in_non_relevant"]),
        "|Cat(Relevant)|":len(community_info["category_in_relevant"]),
        "|Cat(Non Relevant)|":len(community_info["category_in_non_relevant"]),
        "Mean Sim between Relevant":community_info["mean_sim_relevant"],
        "Mean Sim between Non Relevant":community_info["mean_sim_non_relevant"],
        "Mean Sim between Relevant -Non Relevant":community_info["mean_sim_rel_non_rel"],
    }
    records.append(record)
report=pd.DataFrame.from_records(records)

report.sort_values(by="|Colls(Relevant)|",ascending=False)[:40]

Per ogni community:
- prendiamo i copiati e ne contiamo le occorrenze di collection
- vediamo se la collection più frequente è dominante -> vuol dire che in generale ci si ispira ad una sola collection

{Sorare: 90, Collectionx: 5, CollectionY:3, CollectionZ: 2} il 90% delle "ispirazioni" in questa community è su Sorare, quindi è dominante
Se questo vale per la maggior parte delle community possiamo affermare che in media ci si ispira solo ad una collection

In [ ]:
for c,community in df.iterrows():
    copiati=community["most_inspirational_nft"]
    colls=[eval(nft)[0] for nft in copiati]
    colls,freqs=np.unique(colls,return_counts=True)
    freqs=freqs/freqs.sum()
    coll2ptc={colls[i]:freqs[i] for i in range(len(colls))}
    print(coll2ptc)
    